# E5 bonus — GRPO reward IoU từ checkpoint SFT

Chọn GPU T4, bật Internet. Nếu T4 x2, mã chỉ dùng GPU 0.
**Add Input hai nguồn:** dataset road-damage gốc và **Output của notebook E0–E4 đã chạy**.
Output cần chứa `day10/work/e3/adapter/` và `day10/work/e3/projector.pt`.
ZIP `lab_day10_results.zip` không có trọng số, không dùng thay Output checkpoint.

Notebook chỉ chạy E5 (60 bước), không huấn luyện lại E0–E4. Baseline chọn theo validation
đã lưu, thường là E3. Reward được kiểm thử trước khi nạp GPU model.
Save Version → Save & Run All; tải `lab_day10_e5_results.zip` ở Output để phân tích bonus.


## 0. Cấu hình

In [ ]:
SFT_SOURCE = None  # tự tìm Output cũ; hoặc "/kaggle/input/.../day10"
DATA_SOURCE = None  # tự tìm dataset gốc; hoặc root có images/ và labels-YOLO/
MAX_STEPS = 60      # 2 để smoke test; bản 2 bước không dùng làm bằng chứng hoàn thành
MAX_COMPLETION_LENGTH = 128
LEARNING_RATE = 5e-6


## 1. Khôi phục mã và cài TRL 1.9.2

In [ ]:
import base64, io, json, os, subprocess, sys, zipfile
from pathlib import Path
PROJECT = Path("/kaggle/working/day10_e5")
PROJECT.mkdir(parents=True, exist_ok=True)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
os.chdir("/kaggle/working")
PAYLOAD = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    archive.extractall(PROJECT)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # before any torch import, one physical T4
os.environ["HF_HOME"] = "/tmp/day10_hf_cache"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
config = {"input_root": "/kaggle/input", "sft_source": SFT_SOURCE, "data_source": DATA_SOURCE,
          "max_steps": MAX_STEPS, "max_completion_length": MAX_COMPLETION_LENGTH,
          "learning_rate": LEARNING_RATE}
if not isinstance(MAX_STEPS, int) or MAX_STEPS <= 0 or MAX_COMPLETION_LENGTH <= 0 or LEARNING_RATE <= 0:
    raise ValueError("Steps, completion length and learning rate must be positive.")
(PROJECT / "e5_run_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")


In [ ]:
import importlib.metadata as metadata
subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run([sys.executable, "-c",
    "import torch,torchvision; assert torch.cuda.is_available(); "
    "print('Visible GPUs:',torch.cuda.device_count()); assert torch.cuda.device_count()==1; "
    "assert tuple(map(int,torch.__version__.split('+')[0].split('.')[:2])) >= (2,4)"], check=True)
constraints = []
for name in ("torch", "torchvision", "torchaudio"):
    try:
        constraints.append(name + "==" + metadata.version(name))
    except metadata.PackageNotFoundError:
        pass
lock = PROJECT / "torch_constraints.txt"
lock.write_text(chr(10).join(constraints) + chr(10), encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT / "requirements-e5.txt"),
               "-c", str(lock)], check=True)
try:
    metadata.version("torchao")
except metadata.PackageNotFoundError:
    pass
else:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
# REPORT.md is completed; exclude only the teacher test requiring an unfilled template.
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "e5_checks",
               "-k", "not report_template_has_student_markers_and_ten_questions"], cwd=PROJECT, check=True)


## 2. Reward IoU — mã và kiểm thử

In [ ]:
"""Pure Python IoU reward specified by GUIDE step 9; no GPU or TRL needed."""
import json
import math
from pathlib import Path

from scripts.score import parse_output, iou_xyxy


def completion_text(completion):
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        parts = []
        for message in completion:
            if not isinstance(message, dict) or message.get("role", "assistant") != "assistant":
                continue
            content = message.get("content", "")
            if isinstance(content, str):
                parts.append(content)
            elif isinstance(content, list):
                parts.extend(p.get("text", "") for p in content
                             if isinstance(p, dict) and p.get("type") == "text")
        return "".join(parts)
    return ""


def reward_details(completion, gt_boxes):
    """GT uses label/bbox in norm1000, same floating-point geometry as the scorer."""
    raw = completion_text(completion)
    pred = parse_output(raw)
    if pred is None:
        return {"reward": 0.0, "parse_valid": False, "n_pred": 0, "n_gt": len(gt_boxes),
                "empty": False, "matched_pairs": 0, "sum_iou": 0.0, "giant_boxes": 0}
    pairs = sorted(((iou_xyxy(g["bbox"], p["bbox"]), i, j)
                    for i, g in enumerate(gt_boxes) for j, p in enumerate(pred)
                    if g["label"] == p["label"]), reverse=True)
    used_gt, used_pred, total_iou = set(), set(), 0.0
    for value, i, j in pairs:
        if value < 0.5:
            break
        if i in used_gt or j in used_pred:
            continue
        used_gt.add(i)
        used_pred.add(j)
        total_iou += value
    reward = 1.0 if not gt_boxes and not pred else 0.2 + 0.8 * total_iou / max(len(gt_boxes), len(pred), 1)
    giant = sum((p["bbox"][2] - p["bbox"][0]) * (p["bbox"][3] - p["bbox"][1]) >= 250000
                for p in pred)
    assert math.isfinite(reward) and 0 <= reward <= 1
    return {"reward": reward, "parse_valid": True, "n_pred": len(pred), "n_gt": len(gt_boxes),
            "empty": not pred, "matched_pairs": len(used_gt), "sum_iou": total_iou,
            "giant_boxes": giant}


def iou_reward(completion, gt_boxes):
    return reward_details(completion, gt_boxes)["reward"]


class RewardLogger:
    """TRL callable; stores rollouts locally and logs scalar hacking indicators."""
    __name__ = "iou_reward"

    def __init__(self, path, num_generations=4, max_completion_length=128):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.num_generations = num_generations
        self.max_completion_length = max_completion_length

    def __call__(self, completions, gt_boxes, image_id=None, completion_ids=None, **kwargs):
        if len(completions) != len(gt_boxes):
            raise ValueError("GT/completion batch lengths differ")
        ids = image_id if image_id is not None else [None] * len(completions)
        if len(ids) != len(completions):
            raise ValueError("image_id/completion batch lengths differ")
        details = [reward_details(c, g) for c, g in zip(completions, gt_boxes)]
        state = kwargs.get("trainer_state")
        step = getattr(state, "global_step", None)
        with self.path.open("a", encoding="utf-8") as log:
            for i, (c, d) in enumerate(zip(completions, details)):
                length = len(completion_ids[i]) if completion_ids is not None else None
                record = {"trainer_step": step, "image_id": ids[i], "raw": completion_text(c),
                          "completion_tokens": length,
                          "hit_token_limit": length is not None and length >= self.max_completion_length,
                          **d}
                log.write(json.dumps(record, ensure_ascii=False) + "\n")
        log_metric = kwargs.get("log_metric")
        if log_metric and details:
            n = len(details)
            log_metric("reward_audit/parse_valid", sum(d["parse_valid"] for d in details) / n)
            log_metric("reward_audit/empty_on_nonempty_gt", sum(d["empty"] and d["n_gt"] > 0 for d in details) / n)
            log_metric("reward_audit/giant_box_image_rate", sum(d["giant_boxes"] > 0 for d in details) / n)
        return [d["reward"] for d in details]


In [ ]:
# Công thức đề: JSON hỏng=0; GT và pred rỗng=1;
# còn lại 0.2 + 0.8 * sum(matched IoU) / max(n_gt,n_pred,1).
gt = [{"label":"pothole", "bbox":[100,100,200,200]}]
perfect = '[{"label":"pothole","bbox_2d":[100,100,200,200]}]'
print("Perfect:", iou_reward(perfect, gt))
print("Malformed:", iou_reward("not JSON", gt))
print("Empty on positive GT:", iou_reward("[]", gt))
print("Huge box:", iou_reward('[{"label":"pothole","bbox_2d":[0,0,1000,1000]}]', gt))


## 3. Kiểm tra checkpoint và split; không chọn bằng test

In [ ]:
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
import kaggle_e5
def run_task(task):
    log_path = PROJECT / "report/logs" / f"e5_{task}.txt"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with log_path.open("w", encoding="utf-8") as log:
            process = subprocess.Popen([sys.executable, "-u", "kaggle_e5.py", "--task", task],
                cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, encoding="utf-8", errors="replace", bufsize=1)
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, process.args)
    finally:
        kaggle_e5.export_results()
run_task("prepare")
print((PROJECT / "report/e5_warm_start.json").read_text(encoding="utf-8"))


## 4. Huấn luyện GRPO 60 bước

`num_generations=4`, generation batch=4, train microbatch=1 và accumulation=4.
Tiếp tục LoRA đã học; projector SFT được nạp nguyên trạng và đóng băng.
`T4GRPOTrainer` giữ chính xác fp32 adapter khi TRL khởi tạo và autocast fp16 cho VLM.
Log reward/output trong `work/e5/rollouts.jsonl`; checkpoint mỗi 20 bước.


In [ ]:
"""Compatibility for TRL 1.9.2's QLoRA/VLM path on a single fp16 T4."""
from contextlib import nullcontext
import torch
from trl import GRPOTrainer


class T4GRPOTrainer(GRPOTrainer):
    def __init__(self, model, *args, **kwargs):
        # TRL 1.9.2 unconditionally converts quantized-model trainables to BF16.
        # A dtype conversion back alone would lose SFT precision; restore exact values.
        original = {n: p.detach().float().cpu().clone()
                    for n, p in model.named_parameters() if p.requires_grad}
        super().__init__(model, *args, **kwargs)
        restored = dict(self.model.named_parameters())
        for n, value in original.items():
            if n not in restored or not restored[n].requires_grad:
                raise RuntimeError(f"Trainer changed trainable parameter identity: {n}")
            restored[n].data = value.to(device=restored[n].device, dtype=torch.float32)
        verified = all(torch.equal(restored[n].detach().cpu(), value) for n, value in original.items())
        if not verified:
            raise RuntimeError("SFT initialization was not preserved exactly.")
        self.fp32_preservation = {"verified": verified, "trainable_tensors": len(original),
                                  "trainable_params": sum(v.numel() for v in original.values()),
                                  "precision": "fp32 trainables; fp16 CUDA autocast"}

    def _precision_context(self):
        if self.accelerator.device.type == "cuda" and self.args.fp16:
            return torch.autocast("cuda", dtype=torch.float16)
        return nullcontext()

    def _generate_single_turn(self, *args, **kwargs):
        # Generation is outside Trainer.compute_loss_context_manager. The frozen
        # FP32 projector still needs autocast with the FP16 vision backbone.
        with self._precision_context():
            return super()._generate_single_turn(*args, **kwargs)

    def _get_per_token_logps_and_entropies(self, *args, **kwargs):
        # Covers both the detached old-policy logprobs and the training forward.
        with self._precision_context():
            return super()._get_per_token_logps_and_entropies(*args, **kwargs)


In [ ]:
run_task("train")
print((PROJECT / "work/e5/run.json").read_text(encoding="utf-8"))


## 5. Nạp lại E5, suy luận val/test, so sánh SFT và reward hacking

In [ ]:
# Greedy evaluation uses the SAME 512-token limit as the saved SFT predictions.
run_task("evaluate")
print((PROJECT / "report/e5_comparison.json").read_text(encoding="utf-8"))


## 6. Tải bằng chứng bonus

In [ ]:
from IPython.display import FileLink, display
display(FileLink("lab_day10_e5_results.zip"))
print("Tải ZIP trong Output của phiên Save Version và gửi lại để cập nhật báo cáo.")
print("Adapter + projector E5 nằm riêng ở day10_e5/work/e5/. Không commit trọng số.")
